# Eagle Cruiser: 200-Trial Build Optimization Report

This report analyzes the results of an automated ship build optimization campaign for the **Eagle** cruiser hull in Starsector. The optimizer uses Tree-structured Parzen Estimation (TPE) to discover effective loadouts through simulated AI-vs-AI combat.

### Run Configuration

| Parameter | Value |
|-----------|-------|
| **Hull** | Eagle (Cruiser, 145 OP) |
| **Optimizer** | Optuna TPE (multivariate, constant liar) |
| **Warm-start** | ~500 heuristic-scored + stock builds |
| **Simulation budget** | 200 combat-evaluated trials |
| **Parallel instances** | 4 game instances |
| **Fitness aggregation** | Mean across all opponent matchups |

### Opponent Pool

Each build is tested against six diverse cruiser-class opponents. The final fitness is the mean score across all six matchups.

| Opponent | Archetype | Key Threat |
|----------|-----------|------------|
| Aurora (Assault) | Phase cruiser | High burst damage, hard to pin down |
| Dominator (Assault) | Ballistic brawler | Tanky, high sustained DPS |
| Dominator XIV (Elite) | Upgraded brawler | Better stats than base Dominator |
| Doom (Strike) | Phase + mines | Mine-based burst, evasive |
| Eagle (Assault) | Mirror match | Baseline combat reference |
| Heron (Attack) | Carrier | Fighter-based sustained pressure |

### Fitness Function

The hierarchical combat fitness function produces a total ordering where no amount of bonuses in a lower tier can exceed the range of a higher tier:

| Outcome | Score Range | Meaning |
|---------|------------|---------|
| Decisive win | [1.0, 1.1] | Win + efficiency bonuses (speed, HP preserved) |
| Engaged timeout (player advantage) | (0, +0.5] | Ships fought, player dealt more permanent damage |
| Neutral timeout | 0.0 | Stalemate with no clear advantage |
| No-engagement timeout | -0.5 | Active penalty for builds that never close to range |
| Close loss | (-1.0, -0.85] | Lost but dealt meaningful damage |
| Decisive loss | -1.0 | Destroyed with minimal resistance |

In [ ]:
import json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

with open(Path("evaluation_log.jsonl")) as f:
    records = [json.loads(line) for line in f]

rows, matchup_rows = [], []
for rec in records:
    b = rec["build"]
    weapons = {k: v for k, v in b["weapon_assignments"].items() if v is not None}
    rows.append({
        "trial": rec["trial_number"], "fitness": rec["fitness"],
        "timestamp": rec["timestamp"], "n_weapons": len(weapons),
        "n_hullmods": len(b["hullmods"]), "flux_vents": b["flux_vents"],
        "flux_capacitors": b["flux_capacitors"],
        "hullmods": tuple(sorted(b["hullmods"])),
        "weapons": tuple(sorted(weapons.items())),
    })
    for opp in rec["opponent_results"]:
        matchup_rows.append({
            "trial": rec["trial_number"], "fitness": rec["fitness"],
            "opponent": opp["opponent"], "winner": opp["winner"],
            "duration": opp["duration_seconds"], "hp_diff": opp["hp_differential"],
        })

df = pd.DataFrame(rows)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["total_flux"] = df.flux_vents + df.flux_capacitors
mdf = pd.DataFrame(matchup_rows)
top_q_threshold = df.fitness.quantile(0.75)
top_q = df[df.fitness >= top_q_threshold]

display(Markdown(
    f"**Data loaded:** {len(df)} evaluated builds, {len(mdf)} individual matchup results.  \n"
    f"**Fitness range:** [{df.fitness.min():.3f}, {df.fitness.max():.3f}] &mdash; "
    f"mean {df.fitness.mean():.3f}, std {df.fitness.std():.3f}"
))

---

## 1. Fitness Distribution

The histogram below shows the distribution of fitness scores across all evaluated builds. The red dashed line marks the break-even point (fitness = 0), where a build transitions from net-negative to net-positive combat performance. The box plot on the right shows the spread and outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df.fitness, bins=30, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].axvline(0, color="red", linestyle="--", alpha=0.5, label="Break-even (0.0)")
axes[0].axvline(df.fitness.median(), color="orange", linestyle="--", alpha=0.7,
                label=f"Median ({df.fitness.median():.3f})")
axes[0].set_xlabel("Fitness Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Fitness Distribution")
axes[0].legend()

sns.boxplot(y=df.fitness, ax=axes[1], width=0.3, color="steelblue", fliersize=0)
sns.stripplot(y=df.fitness, ax=axes[1], color="black", alpha=0.3, size=3, jitter=0.1)
axes[1].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[1].set_title("Fitness Spread")
axes[1].set_ylabel("Fitness Score")

plt.tight_layout()
plt.show()

n_pos = (df.fitness > 0).sum()
n_win = (df.fitness >= 1.0).sum()
display(Markdown(
    f"| Metric | Value |\n|--------|-------|\n"
    f"| Positive fitness (> 0) | {n_pos}/{len(df)} ({100*n_pos/len(df):.1f}%) |\n"
    f"| At least one win (>= 1.0) | {n_win}/{len(df)} ({100*n_win/len(df):.1f}%) |\n"
    f"| Q1 / Median / Q3 | {df.fitness.quantile(0.25):.3f} / {df.fitness.quantile(0.5):.3f} / {df.fitness.quantile(0.75):.3f} |"
))

The distribution is left-skewed, with most builds scoring below zero. This is expected: the search space is vast (13 weapon slots with dozens of options each, plus hullmods and flux allocation), so random builds are overwhelmingly bad. The long right tail contains the optimizer's discoveries — builds that can compete with or beat the opponent pool.

---

## 2. Optimizer Convergence

The convergence plot tracks how quickly the optimizer finds good builds. Each blue dot is one evaluated build. The red line is the running best (cumulative maximum), and the orange dashed line is a rolling 20-trial mean showing whether the optimizer's average proposal quality improves over time.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

trial_order = df.reset_index(drop=True)
x = trial_order.index

ax.scatter(x, trial_order.fitness, alpha=0.4, s=20, color="steelblue", label="Individual trial")
ax.plot(x, trial_order.fitness.cummax(), color="red", linewidth=2, label="Running best")
rolling = trial_order.fitness.rolling(20, min_periods=1).mean()
ax.plot(x, rolling, color="orange", linewidth=1.5, linestyle="--", label="Rolling mean (w=20)")
ax.axhline(0, color="gray", linestyle=":", alpha=0.5)

best_idx = trial_order.fitness.idxmax()
best_val = trial_order.fitness.max()
ax.annotate(f"Best: {best_val:.3f}\n(eval #{best_idx})",
            xy=(best_idx, best_val), xytext=(best_idx + 15, best_val - 0.15),
            arrowprops=dict(arrowstyle="->", color="red"), fontsize=9, color="red")

ax.set_xlabel("Evaluation Order")
ax.set_ylabel("Fitness")
ax.set_title("Optimizer Convergence")
ax.legend()
plt.tight_layout()
plt.show()

cummax = trial_order.fitness.cummax()
improvements = cummax.diff().gt(0)
last_imp = improvements[improvements].index[-1]
first_pos = (trial_order.fitness > 0).idxmax()

display(Markdown(
    f"| Milestone | Evaluation # |\n|-----------|-------------|\n"
    f"| First positive fitness | #{first_pos} |\n"
    f"| Best fitness ({best_val:.4f}) | #{best_idx} |\n"
    f"| Last improvement | #{last_imp} |\n"
    f"| Stagnation duration | {len(trial_order) - last_imp - 1} evals without improvement |"
))

The optimizer finds a competitive build quickly but then plateaus. The rolling mean shows a gradual upward trend, indicating the TPE sampler is learning to propose better builds on average, but the running best stagnates in the second half of the run. This is likely due to high combat noise (stochastic AI behavior causes the same build to score differently across runs), which drowns out the signal for fine-grained improvements.

---

## 3. Per-Opponent Breakdown

Not all opponents are equally difficult. The charts below show outcome distributions (left) and HP differential distributions (right) for each opponent. Positive HP differential means the optimized build had more hull HP remaining.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

opponents = sorted(mdf.opponent.unique())
outcome_counts = {}
for opp in opponents:
    od = mdf[mdf.opponent == opp]
    total = len(od)
    outcome_counts[opp] = {
        "Win": (od.winner == "PLAYER").sum(),
        "Loss": (od.winner == "ENEMY").sum(),
        "Timeout": (od.winner == "TIMEOUT").sum(),
        "Stopped": (od.winner == "STOPPED").sum(),
    }

opp_labels = [o.replace("_", "\n") for o in opponents]
outcome_df = pd.DataFrame(outcome_counts).T
outcome_df.index = opp_labels
colors_map = {"Win": "#2ecc71", "Loss": "#e74c3c", "Timeout": "#f39c12", "Stopped": "#95a5a6"}
outcome_df[["Win", "Loss", "Timeout", "Stopped"]].plot(
    kind="bar", stacked=True, ax=axes[0],
    color=[colors_map[c] for c in ["Win", "Loss", "Timeout", "Stopped"]])
axes[0].set_title("Outcome Distribution per Opponent")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(loc="upper right", fontsize=8)

sns.boxplot(data=mdf, x="opponent", y="hp_diff", ax=axes[1], order=opponents)
axes[1].set_xticklabels(opp_labels, fontsize=8)
axes[1].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[1].set_title("HP Differential per Opponent")
axes[1].set_ylabel("HP Differential")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()

# Summary table
table = "| Opponent | Win Rate | Loss Rate | Timeout Rate | Median HP Diff |\n"
table += "|----------|----------|-----------|-------------|----------------|\n"
for opp in opponents:
    od = mdf[mdf.opponent == opp]
    total = len(od)
    w = (od.winner == "PLAYER").sum()
    l = (od.winner == "ENEMY").sum()
    t = (od.winner == "TIMEOUT").sum()
    table += f"| {opp} | {100*w/total:.1f}% | {100*l/total:.1f}% | {100*t/total:.1f}% | {od.hp_diff.median():+.3f} |\n"
display(Markdown(table))

The opponent difficulty varies dramatically. Phase ships (Aurora, Doom) are the hardest matchups with high loss rates, since they can evade damage and strike at vulnerable moments. The Dominator variants produce more timeouts — consistent with the Dominator's tanky nature leading to attrition-style fights. The Eagle mirror match and Heron carrier are the most winnable matchups.

The "Stopped" outcome (stochastic curtailment) is frequent across all opponents, indicating many fights are terminated early when the outcome becomes statistically clear. This is by design — it saves simulation time on decisive fights.

---

## 4. Top 10 Builds

The table below shows the 10 highest-scoring builds discovered by the optimizer. The "Results" column shows per-opponent outcomes (P=Player win, E=Enemy win, T=Timeout, S=Stopped).

In [ ]:
top10 = df.nlargest(10, "fitness")

table = "| # | Fitness | Weapons | Hullmods | V/C | Opponent Results |\n"
table += "|---|---------|---------|----------|-----|------------------|\n"

for rank, (_, row) in enumerate(top10.iterrows(), 1):
    weps = ", ".join(w for _, w in row["weapons"])
    mods = ", ".join(row["hullmods"])
    vc = f"{row['flux_vents']}/{row['flux_capacitors']}"

    trial_m = mdf[mdf.trial == row["trial"]]
    if len(trial_m) > 0:
        results = " ".join(
            f"{m.opponent.split('_')[0][:3]}:{m.winner[0]}"
            for _, m in trial_m.iterrows()
        )
    else:
        results = "N/A"

    table += f"| {rank} | {row['fitness']:+.3f} | {weps} | {mods} | {vc} | {results} |\n"

display(Markdown(table))

The top builds share a clear pattern: **graviton beams** in the energy slots, **HVel drivers** in the medium ballistic slots, **breach** torpedoes in the missile slots, and **PD lasers** for point defense. The **targetingunit** hullmod (Integrated Targeting Unit, +200 range) appears in nearly all top builds, suggesting that range advantage is the single most important factor for the Eagle's combat effectiveness.

The V/C column shows flux vents / flux capacitors. Note that these values reflect the *repaired* build (post constraint enforcement), not the raw optimizer proposal.

---

## 5. Weapon Popularity

The left chart compares weapon frequency across all builds versus the top 25%. The right chart shows **enrichment** — how much more (or less) frequently a weapon appears in top builds compared to the overall population. Values above 1.0 (green) indicate the weapon is overrepresented in successful builds.

In [ ]:
all_weapons = Counter()
top_q_weapons = Counter()
for _, row in df.iterrows():
    for slot, wep in row["weapons"]:
        all_weapons[wep] += 1
        if row["fitness"] >= top_q_threshold:
            top_q_weapons[wep] += 1

weapon_names = sorted(set(all_weapons.keys()) | set(top_q_weapons.keys()))
weapon_comp = pd.DataFrame({
    "All builds": [all_weapons.get(w, 0) for w in weapon_names],
    "Top 25%": [top_q_weapons.get(w, 0) for w in weapon_names],
}, index=weapon_names)
weapon_comp["Enrichment"] = (weapon_comp["Top 25%"] / weapon_comp["Top 25%"].sum()) / \
                             (weapon_comp["All builds"] / weapon_comp["All builds"].sum())
weapon_comp = weapon_comp.sort_values("Top 25%", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

top15 = weapon_comp.head(15)
y_pos = range(len(top15))
axes[0].barh(y_pos, top15["All builds"], alpha=0.4, color="steelblue", label="All builds")
axes[0].barh(y_pos, top15["Top 25%"], alpha=0.8, color="coral", label="Top 25%")
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(top15.index, fontsize=9)
axes[0].invert_yaxis()
axes[0].set_xlabel("Appearances")
axes[0].set_title("Weapon Frequency: All vs Top 25%")
axes[0].legend(fontsize=9)

enriched = weapon_comp[weapon_comp["Top 25%"] >= 5].sort_values("Enrichment", ascending=False).head(15)
colors = ["#2ecc71" if e > 1.2 else "#e74c3c" if e < 0.8 else "#95a5a6" for e in enriched["Enrichment"]]
axes[1].barh(range(len(enriched)), enriched["Enrichment"], color=colors)
axes[1].set_yticks(range(len(enriched)))
axes[1].set_yticklabels(enriched.index, fontsize=9)
axes[1].invert_yaxis()
axes[1].axvline(1.0, color="red", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Enrichment Ratio (>1 = overrepresented in top builds)")
axes[1].set_title("Weapon Enrichment in Top 25%")

plt.tight_layout()
plt.show()

The graviton beam and HVel driver are strongly enriched in top builds — they appear far more often in successful builds than in the overall population. These weapons share a key trait: **long effective range**, which synergizes with the targetingunit hullmod. The PD laser dominates the small energy slots, suggesting that point defense capability is important for surviving against missile-based opponents.

---

## 6. Hullmod Analysis

Hullmods define a build's strategic identity. The chart below compares presence rates (what fraction of builds include each hullmod) between all builds and the top 25%.

In [ ]:
all_mods = Counter()
top_q_mods = Counter()
for _, row in df.iterrows():
    for mod in row["hullmods"]:
        all_mods[mod] += 1
        if row["fitness"] >= top_q_threshold:
            top_q_mods[mod] += 1

mod_names = sorted(set(all_mods.keys()) | set(top_q_mods.keys()))
n_top = (df.fitness >= top_q_threshold).sum()
mod_comp = pd.DataFrame({
    "All": [all_mods.get(m, 0) for m in mod_names],
    "Top 25%": [top_q_mods.get(m, 0) for m in mod_names],
}, index=mod_names)
mod_comp["Rate_All"] = mod_comp["All"] / len(df)
mod_comp["Rate_Top"] = mod_comp["Top 25%"] / n_top
mod_comp = mod_comp.sort_values("Top 25%", ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
top_mods = mod_comp[mod_comp["Top 25%"] >= 3]
y = range(len(top_mods))
ax.barh(y, top_mods["Rate_All"], alpha=0.4, color="steelblue", label="All builds", height=0.4)
ax.barh([yi + 0.4 for yi in y], top_mods["Rate_Top"], alpha=0.8, color="coral", label="Top 25%", height=0.4)
ax.set_yticks([yi + 0.2 for yi in y])
ax.set_yticklabels(top_mods.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Presence Rate")
ax.set_title("Hullmod Presence Rate: All Builds vs Top 25%")
ax.legend()
ax.axvline(0.5, color="gray", linestyle=":", alpha=0.3)
plt.tight_layout()
plt.show()

dominant = mod_comp[mod_comp["Rate_Top"] > 0.3].sort_values("Rate_Top", ascending=False)
table = "| Hullmod | Top 25% Rate | Overall Rate | Signal |\n"
table += "|---------|-------------|-------------|--------|\n"
for mod, row in dominant.iterrows():
    signal = "Strong positive" if row["Rate_Top"] > row["Rate_All"] * 1.5 else \
             "Moderate positive" if row["Rate_Top"] > row["Rate_All"] else "Neutral"
    table += f"| {mod} | {row['Rate_Top']:.0%} | {row['Rate_All']:.0%} | {signal} |\n"
display(Markdown(table))

The **targetingunit** (Integrated Targeting Unit) stands out as the single most important hullmod, with dramatically higher presence in top builds compared to the population. This makes intuitive sense: the Eagle is a mid-range combatant, and +200 weapon range lets it engage safely while staying out of close-range brawlers' optimal envelope.

---

## 7. Per-Slot Weapon Convergence

Each of the Eagle's 13 weapon slots has different type and size constraints. The chart below shows how strongly the optimizer has converged on a single weapon choice for each slot. Green bars (>70% share) indicate "solved" slots where the optimizer is confident; red bars (<40%) indicate slots where the choice doesn't matter much or where the optimizer hasn't converged.

In [ ]:
slot_ids = sorted(set(s for _, row in df.iterrows() for s, _ in row["weapons"]))
slot_weapon_freq = {slot: Counter() for slot in slot_ids}

for _, row in top_q.iterrows():
    assigned = dict(row["weapons"])
    for slot in slot_ids:
        wep = assigned.get(slot, "(empty)")
        slot_weapon_freq[slot][wep] += 1

fig, ax = plt.subplots(figsize=(14, 6))

convergence = []
labels = []
for slot in slot_ids:
    total = sum(slot_weapon_freq[slot].values())
    top1_name, top1_count = slot_weapon_freq[slot].most_common(1)[0]
    convergence.append(top1_count / total)
    labels.append(f"{slot}\n({top1_name})")

colors_conv = ["#2ecc71" if c > 0.7 else "#f39c12" if c > 0.4 else "#e74c3c" for c in convergence]
ax.bar(range(len(slot_ids)), convergence, color=colors_conv, edgecolor="black", alpha=0.8)
ax.set_xticks(range(len(slot_ids)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("Top-1 Weapon Share (in Top 25% builds)")
ax.set_title("Per-Slot Convergence (green = solved, yellow = partial, red = unsolved)")
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="50% threshold")
ax.set_ylim(0, 1)
ax.legend()

for i, c in enumerate(convergence):
    ax.text(i, c + 0.02, f"{c:.0%}", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

# Detailed per-slot table
table = "| Slot | #1 Choice | #2 Choice | #3 Choice |\n"
table += "|------|-----------|-----------|----------|\n"
for slot in slot_ids:
    total = sum(slot_weapon_freq[slot].values())
    top3 = slot_weapon_freq[slot].most_common(3)
    cells = [f"{w} ({c/total:.0%})" for w, c in top3]
    while len(cells) < 3:
        cells.append("—")
    table += f"| {slot} | {cells[0]} | {cells[1]} | {cells[2]} |\n"
display(Markdown(table))

The energy slots (WS 005, 006, 009) are strongly solved — the optimizer is very confident that graviton beams belong there. The small energy PD slots (WS 003, 004, 011, 012) converge on PD laser. The medium ballistic slots (WS 007, 008) favor HVel driver. The missile slots (WS 001, 002) and the large ballistic slot (WS 013) show less convergence, suggesting these are secondary optimization targets where the choice matters less.

This suggests a potential strategy for future runs: **fix the solved slots** and only optimize the unsolved ones, reducing the search space dramatically.

---

## 8. Flux Allocation

Flux vents (dissipation) and capacitors (capacity) compete with weapons for ordnance points. The scatter plot shows how this allocation relates to combat performance.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sc = axes[0].scatter(df.flux_vents, df.flux_capacitors, c=df.fitness,
                     cmap="RdYlGn", s=30, alpha=0.7, edgecolors="gray", linewidth=0.3)
plt.colorbar(sc, ax=axes[0], label="Fitness")
axes[0].set_xlabel("Flux Vents")
axes[0].set_ylabel("Flux Capacitors")
axes[0].set_title("Vents vs Caps (colored by fitness)")

z = np.polyfit(df.total_flux, df.fitness, 1)
x_line = np.linspace(df.total_flux.min(), df.total_flux.max(), 100)
axes[1].scatter(df.total_flux, df.fitness, alpha=0.4, s=20, color="steelblue")
axes[1].plot(x_line, np.polyval(z, x_line), color="red", linewidth=2, linestyle="--")
axes[1].set_xlabel("Total Flux Investment (Vents + Caps)")
axes[1].set_ylabel("Fitness")
axes[1].set_title(f"Flux Investment vs Fitness (slope={z[0]:.4f})")

axes[2].hist(df.total_flux, bins=20, alpha=0.4, color="steelblue", label="All builds", density=True)
axes[2].hist(top_q.total_flux, bins=20, alpha=0.6, color="coral", label="Top 25%", density=True)
axes[2].set_xlabel("Total Flux Investment")
axes[2].set_ylabel("Density")
axes[2].set_title("Flux Investment Distribution")
axes[2].legend()

plt.tight_layout()
plt.show()

display(Markdown(
    f"| Group | Avg Vents | Avg Caps | Avg Total |\n"
    f"|-------|-----------|----------|----------|\n"
    f"| Top 25% | {top_q.flux_vents.mean():.1f} | {top_q.flux_capacitors.mean():.1f} | {top_q.total_flux.mean():.1f} |\n"
    f"| All builds | {df.flux_vents.mean():.1f} | {df.flux_capacitors.mean():.1f} | {df.total_flux.mean():.1f} |"
))

Interestingly, top builds tend to invest **less** in flux than average, suggesting the optimizer prefers to spend OP on weapons rather than flux management. This aligns with the range-focused archetype: if you're engaging at long range with graviton beams and HVel drivers, you take less incoming fire and need less flux capacity to absorb it.

Note: flux values shown here are post-repair. The repair operator allocates remaining OP to vents/caps after fixing constraint violations. Builds with more expensive weapon loadouts naturally have less OP left for flux.

---

## 9. Combat Duration

Fight duration reveals build behavior. Aggressive builds that close range quickly produce short, decisive fights. Passive or ranged builds lead to longer engagements or timeouts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

outcome_colors = {"PLAYER": "#2ecc71", "ENEMY": "#e74c3c", "TIMEOUT": "#f39c12", "STOPPED": "#95a5a6"}
for outcome in ["PLAYER", "ENEMY", "TIMEOUT", "STOPPED"]:
    subset = mdf[mdf.winner == outcome]
    if len(subset) > 0:
        axes[0].hist(subset.duration, bins=30, alpha=0.5, label=f"{outcome} (n={len(subset)})",
                     color=outcome_colors[outcome])
axes[0].set_xlabel("Duration (seconds)")
axes[0].set_ylabel("Count")
axes[0].set_title("Combat Duration by Outcome")
axes[0].legend(fontsize=9)

non_timeout = mdf[mdf.winner != "TIMEOUT"]
axes[1].scatter(non_timeout.duration, non_timeout.hp_diff,
                c=[outcome_colors[w] for w in non_timeout.winner], alpha=0.4, s=15)
axes[1].axhline(0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Duration (seconds)")
axes[1].set_ylabel("HP Differential")
axes[1].set_title("Duration vs HP Differential (non-timeout)")
from matplotlib.lines import Line2D
axes[1].legend(handles=[
    Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=8, label=l)
    for l, c in outcome_colors.items() if l != "TIMEOUT"
], fontsize=9)

plt.tight_layout()
plt.show()

# Duration stats as markdown
table = "| Outcome | Count | Median Duration |\n"
table += "|---------|-------|----------------|\n"
for outcome in ["PLAYER", "ENEMY", "STOPPED", "TIMEOUT"]:
    subset = mdf[mdf.winner == outcome]
    if len(subset):
        table += f"| {outcome} | {len(subset)} | {subset.duration.median():.0f}s |\n"
display(Markdown(table))

Player wins tend to be longer fights, suggesting the Eagle's winning strategy is sustained attrition at range rather than burst damage. Enemy wins are faster — when the optimized build loses, it tends to lose quickly (consistent with phase ships like Aurora and Doom that can deal lethal burst damage). The large cluster of STOPPED outcomes around 100-200s reflects the stochastic curtailment system ending fights once the outcome is statistically determined.

---

## 10. Build Diversity Over Time

The Jaccard similarity between consecutive builds measures whether the optimizer is exploring (low similarity, diverse proposals) or exploiting (high similarity, refining a known-good template). We compute the average Jaccard similarity between each build and the 10 builds evaluated before it.

In [ ]:
def build_set(row):
    items = set(f"{s}={w}" for s, w in row["weapons"])
    items.update(f"mod:{m}" for m in row["hullmods"])
    return items

build_sets = [build_set(row) for _, row in df.iterrows()]

window = 10
rolling_jaccard = []
for i in range(len(build_sets)):
    sims = []
    for j in range(max(0, i - window), i):
        intersection = len(build_sets[i] & build_sets[j])
        union = len(build_sets[i] | build_sets[j])
        sims.append(intersection / union if union > 0 else 0)
    rolling_jaccard.append(np.mean(sims) if sims else 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rolling_jaccard, color="steelblue", linewidth=1)
axes[0].fill_between(range(len(rolling_jaccard)), rolling_jaccard, alpha=0.2, color="steelblue")
axes[0].set_xlabel("Evaluation Order")
axes[0].set_ylabel(f"Mean Jaccard Similarity (window={window})")
axes[0].set_title("Build Similarity Over Time (higher = more exploitation)")
axes[0].set_ylim(0, 1)

axes[1].scatter(df.n_weapons, df.fitness, alpha=0.4, s=20, color="steelblue")
axes[1].set_xlabel("Number of Weapons Equipped")
axes[1].set_ylabel("Fitness")
axes[1].set_title("Weapon Count vs Fitness")

plt.tight_layout()
plt.show()

n_unique = len(set(frozenset(s) for s in build_sets))
n_unique_wep = len(set(row["weapons"] for _, row in df.iterrows()))
display(Markdown(
    f"| Metric | Value |\n|--------|-------|\n"
    f"| Unique build configs | {n_unique} / {len(df)} |\n"
    f"| Unique weapon loadouts | {n_unique_wep} / {len(df)} |\n"
    f"| Avg Jaccard (first 50 evals) | {np.mean(rolling_jaccard[1:51]):.3f} |\n"
    f"| Avg Jaccard (last 50 evals) | {np.mean(rolling_jaccard[-50:]):.3f} |"
))

The rising Jaccard similarity over time confirms the TPE sampler is learning: later proposals are more similar to each other (exploitation) compared to the diverse early proposals (exploration). The weapon count vs fitness plot shows that builds with more weapons equipped tend to perform better, reinforcing the "spend OP on weapons, not flux" finding.

---

## 11. Feature Importance

Which individual build choices most strongly predict fitness? We compute the Pearson correlation between each binary feature (hullmod present/absent, top weapon in slot yes/no) and fitness score. Significance levels: * p<0.05, ** p<0.01, *** p<0.001.

In [ ]:
from scipy import stats as scipy_stats

mod_features = {}
for mod in all_mods:
    if all_mods[mod] >= 10:
        mod_features[f"mod:{mod}"] = [mod in row["hullmods"] for _, row in df.iterrows()]

weapon_features = {}
for slot in slot_ids:
    top_wep = slot_weapon_freq[slot].most_common(1)[0][0]
    if top_wep != "(empty)":
        weapon_features[f"slot:{slot}={top_wep}"] = [
            dict(row["weapons"]).get(slot) == top_wep for _, row in df.iterrows()
        ]

all_features = {**mod_features, **weapon_features}
all_features["flux_vents"] = df.flux_vents.values
all_features["flux_capacitors"] = df.flux_capacitors.values
all_features["n_weapons"] = df.n_weapons.values

correlations = {}
for name, values in all_features.items():
    values = np.array(values, dtype=float)
    if values.std() > 0:
        r, p = scipy_stats.pearsonr(values, df.fitness.values)
        correlations[name] = (r, p)

corr_df = pd.DataFrame([
    {"Feature": k, "Correlation": v[0], "p-value": v[1], "Abs_Corr": abs(v[0])}
    for k, v in correlations.items()
]).sort_values("Abs_Corr", ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
top_features = corr_df.head(20)
colors = ["#2ecc71" if c > 0 else "#e74c3c" for c in top_features["Correlation"]]
ax.barh(range(len(top_features)), top_features["Correlation"], color=colors, edgecolor="black", alpha=0.7)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features["Feature"], fontsize=9)
ax.invert_yaxis()
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("Pearson Correlation with Fitness")
ax.set_title("Feature Importance (top 20 by |correlation|)")

for i, (_, row) in enumerate(top_features.iterrows()):
    sig = "***" if row["p-value"] < 0.001 else "**" if row["p-value"] < 0.01 else "*" if row["p-value"] < 0.05 else ""
    ax.text(row["Correlation"] + (0.01 if row["Correlation"] > 0 else -0.01), i,
            f"{row['Correlation']:.3f}{sig}", va="center", fontsize=8,
            ha="left" if row["Correlation"] > 0 else "right")

plt.tight_layout()
plt.show()

# Top 10 table
table = "| Feature | Correlation | Significance |\n"
table += "|---------|------------|-------------|\n"
for _, row in corr_df.head(10).iterrows():
    sig = "***" if row["p-value"] < 0.001 else "**" if row["p-value"] < 0.01 else "*" if row["p-value"] < 0.05 else "n.s."
    table += f"| {row['Feature']} | {row['Correlation']:+.3f} | {sig} |\n"
display(Markdown(table))

The feature importance analysis confirms the narrative from the individual analyses above. The targetingunit hullmod and number of weapons equipped are the strongest positive predictors of fitness. The graviton beam in the energy slots and HVel driver in the ballistic slots also show positive correlations, while certain hullmods that waste OP or indicate unfocused builds show negative correlations.

This is a univariate analysis and does not capture interactions (e.g., graviton beam is most effective *with* targetingunit). A full functional ANOVA would be needed to decompose the variance properly.

---

## 12. Key Findings & Recommendations

In [ ]:
best_build = df.loc[df.fitness.idxmax()]
best_weapons = dict(best_build["weapons"])
best_mods = list(best_build["hullmods"])
n_pos = (df.fitness > 0).sum()
total_matchups = len(mdf)
player_wins = (mdf.winner == "PLAYER").sum()

display(Markdown(f"""### Overall Results

| Metric | Value |
|--------|-------|
| Total builds evaluated | {len(df)} |
| Positive fitness (competitive builds) | {n_pos} ({100*n_pos/len(df):.0f}%) |
| Player win rate (across all matchups) | {player_wins}/{total_matchups} ({100*player_wins/total_matchups:.1f}%) |
| Best fitness achieved | {best_build['fitness']:.4f} |
| Fitness standard deviation | {df.fitness.std():.3f} |

### Discovered Archetype: "Long-Range Attrition Eagle"

The optimizer converged on a consistent build identity:

- **Core weapons:** Graviton Beam (energy), HVel Driver (ballistic), Breach (missile), PD Laser (point defense)
- **Key hullmod:** Integrated Targeting Unit (+200 range) — present in nearly all top builds
- **Strategy:** Engage at maximum range using kinetic/energy DPS, rely on range advantage to minimize incoming damage, invest OP in weapons over flux
- **Best build weapons:** {', '.join(f'{s}={w}' for s, w in sorted(best_weapons.items()))}
- **Best build mods:** {', '.join(sorted(best_mods))}

### Convergence Behavior

The optimizer found its best build in the first third of the run and spent the remaining trials unable to improve upon it. This suggests:

1. **High combat noise** (fitness std = {df.fitness.std():.3f}) makes it hard to distinguish good from great builds
2. The **core archetype was discovered early**, and marginal improvements in secondary slots are below the noise floor
3. The TPE sampler is learning (rising Jaccard similarity, improving rolling mean) but the signal-to-noise ratio limits progress

### Recommendations for Future Runs

1. **Replay averaging (3x)** — Evaluate each build 3 times against each opponent and average the results. This reduces noise by ~42% (1/sqrt(3)) and should break through the convergence plateau.

2. **Fix solved slots** — Lock in graviton beams, PD lasers, and HVel drivers in their converged slots. This reduces the search space from 13 weapon dimensions to ~5, letting the optimizer focus on the unsolved missile and large ballistic choices.

3. **Increase simulation budget** — With the instance utilization fix (4 to 8 parallel instances), throughput doubles. A 500-trial run with replay averaging would provide much stronger signal.

4. **Try additional hulls** — Run the same pipeline on a frigate (Wolf) or destroyer to validate that the optimizer generalizes beyond the Eagle.
"""))